In [2]:
# --- Cell 1: Install Dependencies ---
# We force Streamlit 1.40.0 to support the drawing canvas
!pip install -q torch torchvision earthengine-api rasterio segmentation-models-pytorch
!pip install -q streamlit==1.40.0 streamlit-drawable-canvas pyngrok
!pip install -q scikit-learn scikit-image matplotlib pandas tqdm

import torch
import numpy as np
import pandas as pd
import random
import os

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment Ready. Using device: {device}")

Environment Ready. Using device: cuda


In [3]:
# --- Cell 2: Real Data Pipeline (Robust & Corrected) ---
import ee
import numpy as np
import rasterio
import os
import requests
import torch

# --- CONFIGURATION ---
PROJECT_ID = 'nca-deforestation-lab'

try:
    ee.Initialize(project=PROJECT_ID)
    print("GEE Initialized successfully.")
except Exception:
    print("Authenticating...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

def download_rondonia_data():
    filename = "rondonia_stack.tif"

    if os.path.exists(filename):
        os.remove(filename)

    print("Preparing GEE Export (Rondônia, Brazil)...")

    # ROI: 50km x 50km area in Rondonia
    roi = ee.Geometry.Rectangle([-62.5, -10.5, -62.0, -10.0])

    # A. Forest Cover (Updated to v1_12 to avoid deprecation)
    gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
    tree_cover = gfc.select('treecover2000').unmask(0)
    loss_year = gfc.select('lossyear').unmask(0)

    # B. Infrastructure (ESA WorldCover v100 - Built-up Class)
    wc = ee.ImageCollection("ESA/WorldCover/v100").first().select('Map')
    infrastructure = wc.eq(50) # Class 50 = Built-up/Roads

    # --- CRITICAL FIX: REPROJECTION ---
    # GEE cannot calculate 20km kernels at 10m resolution (exceeds 512px limit).
    # We force calculation at 100m, reducing kernel radius to 200px.
    infra_100m = infrastructure.reproject(crs='EPSG:4326', scale=100)

    dist_kernel = ee.Kernel.euclidean(radius=20000, units='meters')
    infra_dist = infra_100m.distance(kernel=dist_kernel).unmask(20000)

    # Normalize (1.0 = On Road, 0.0 = >20km away)
    road_norm = infra_dist.divide(20000).clamp(0, 1).multiply(-1).add(1)

    # C. Elevation
    srtm = ee.Image('USGS/SRTMGL1_003').unmask(0)
    elev_norm = srtm.divide(1000).clamp(0, 1)

    # D. Stack (Input 2015 -> Target 2020)
    def get_forest_at_year(y):
        return tree_cover.gt(50).And(loss_year.eq(0).Or(loss_year.gt(y - 2000)))

    forest_2015 = get_forest_at_year(2015).rename('forest_in')
    forest_2020 = get_forest_at_year(2020).rename('forest_out')

    final_stack = ee.Image.cat([forest_2015, road_norm, elev_norm, forest_2020]).float()

    # E. Download
    print("Requesting download URL...")
    try:
        url = final_stack.getDownloadURL({
            'scale': 100, # 100m resolution
            'crs': 'EPSG:4326',
            'region': roi,
            'format': 'GEO_TIFF'
        })
        print(f"Downloading from: {url}")

        r = requests.get(url, stream=True)

        if r.status_code != 200:
            print(f"Server Error: {r.text[:500]}")
            return

        if 'json' in r.headers.get('Content-Type', ''):
            print(f"API Error: {r.text}")
            return

        with open(filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"Download Success! File size: {os.path.getsize(filename)/1024:.2f} KB")

    except Exception as e:
        print(f"Export Exception: {e}")

# Trigger Download
download_rondonia_data()

Preparing GEE Export (Rondônia, Brazil)...
Requesting download URL...
Download Success! File size: 1447.81 KB


In [4]:
# --- Cell 3: Dataset Class ---
from torch.utils.data import Dataset, DataLoader, random_split
import rasterio
import torch
import numpy as np
import os

class RealDeforestationDataset(Dataset):
    def __init__(self, tiff_path, patch_size=64, stride=32, augment=True):
        self.patch_size = patch_size
        self.augment = augment
        self.patches = []

        if not os.path.exists(tiff_path) or os.path.getsize(tiff_path) < 5000:
            print("Warning: TIF file missing. Please run Cell 2 (Download) first.")
            return

        print("Slicing GeoTIFF into patches...")
        with rasterio.open(tiff_path) as src:
            data = src.read()
            _, H, W = data.shape

            valid = 0
            skipped = 0

            # We iterate through the large image
            for y in range(0, H - patch_size, stride):
                for x in range(0, W - patch_size, stride):
                    patch = data[:, y:y+patch_size, x:x+patch_size]

                    if patch.shape[1] != patch_size or patch.shape[2] != patch_size:
                        continue

                    input_forest = patch[0]
                    target_forest = patch[3]

                    # --- THE FIX IS HERE ---
                    # Old: > 0.02 (Allowed patches with 98% dirt)
                    # New: > 0.15 (Must have at least 15% forest to be useful)
                    has_forest = input_forest.mean() > 0.15

                    # We still want patches where change happened
                    change_magnitude = np.abs(input_forest - target_forest).mean()

                    if has_forest and change_magnitude > 0.01:
                        self.patches.append(torch.from_numpy(patch).float())
                        valid += 1
                    else:
                        skipped += 1

        print(f"Dataset Ready: {valid} quality patches loaded (Skipped {skipped} sparse/empty patches).")

    def __len__(self):
        return len(self.patches) * (4 if self.augment else 1)

    def __getitem__(self, idx):
        base_idx = idx % len(self.patches)
        aug_type = idx // len(self.patches)
        p = self.patches[base_idx].clone()

        if self.augment and aug_type > 0:
            if aug_type == 1: p = torch.flip(p, [2])
            elif aug_type == 2: p = torch.flip(p, [1])
            elif aug_type == 3: p = torch.flip(torch.flip(p, [1]), [2])

        return p[0:3], p[3:4]

# Reload the dataset with the new filter
ds_real = RealDeforestationDataset("rondonia_stack.tif", stride=32, augment=True)

Slicing GeoTIFF into patches...
Dataset Ready: 171 quality patches loaded (Skipped 85 sparse/empty patches).


In [5]:
# --- Cell 4: Scientific Methodology ---
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, jaccard_score
from skimage.measure import label

# 1. TUNED LOSS FUNCTION
class RobustChangeLoss(nn.Module):
    def __init__(self, change_weight=15.0, false_positive_weight=5.0):
        super().__init__()
        self.change_weight = change_weight
        self.fp_weight = false_positive_weight
        self.mse = nn.MSELoss(reduction='none')

    def forward(self, pred, target, input_state):
        pixel_loss = self.mse(pred, target)

        # Masks
        # "Missed Change": Target=0 (Deforested), Pred > 0.1
        missed_change = (target < 0.1) & (input_state > 0.9) & (pred > 0.1)
        # "Hallucination": Target=1 (Forest), Pred < 0.9
        hallucination = (target > 0.9) & (pred < 0.9)

        # Apply aggressive weights
        weights = torch.ones_like(pred)
        weights[missed_change] = self.change_weight
        weights[hallucination] = self.fp_weight

        return (pixel_loss * weights).mean()

# 2. METRICS
def get_scientific_metrics(y_true, y_pred, threshold=0.5):
    yt = (y_true.flatten() > threshold).astype(int)
    yp = (y_pred.flatten() > threshold).astype(int)

    mse = np.mean((y_true - y_pred)**2)
    f1 = f1_score(yt, yp, zero_division=0)
    iou = jaccard_score(yt, yp, zero_division=0)

    # Spatial Structure Check
    true_blobs = label(y_true > threshold, connectivity=2)
    pred_blobs = label(y_pred > threshold, connectivity=2)
    n_true, n_pred = true_blobs.max(), pred_blobs.max()

    if n_true == 0: spatial = 1.0 if n_pred == 0 else 0.0
    else: spatial = 1.0 - min(abs(n_true - n_pred) / n_true, 1.0)

    return {"MSE": mse, "F1": f1, "IoU": iou, "Spatial_Structure": spatial}

In [6]:
# --- Cell 5: Model Architectures ---
import torch.nn.functional as F
from sklearn.ensemble import RandomForestRegressor

# 1. PHYSICS-INFORMED NCA
class ResearchNCA(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.channels = channels
        self.sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)/8.0
        self.sobel_y = self.sobel_x.transpose(2,3)
        self.register_buffer('Kx', self.sobel_x)
        self.register_buffer('Ky', self.sobel_y)

        self.perceive_conv = nn.Conv2d(channels, channels, 3, padding=1)
        self.w1 = nn.Conv2d((channels*4)+2, 128, 1)
        self.w2 = nn.Conv2d(128, channels, 1)

    def perceive(self, state):
        k = state.shape[1]
        x_grad = F.conv2d(state, self.Kx.repeat(k,1,1,1), padding=1, groups=k)
        y_grad = F.conv2d(state, self.Ky.repeat(k,1,1,1), padding=1, groups=k)
        learned = self.perceive_conv(state)
        return torch.cat([state, x_grad, y_grad, learned], dim=1)

    def get_slope(self, state):
        elev = state[:, 2:3]
        dx = F.conv2d(elev, self.Kx, padding=1)
        dy = F.conv2d(elev, self.Ky, padding=1)
        return torch.sqrt(dx**2 + dy**2)

    def forward(self, x, steps=32, use_physics=False):
        forest_init = x[:, 0:1]
        static = x[:, 1:3]
        b, _, h, w = forest_init.shape
        hidden = torch.zeros(b, self.channels - 1, h, w, device=x.device)
        state = torch.cat([forest_init, hidden], dim=1)

        slope_map = self.get_slope(x)
        if slope_map.max() > 0: slope_map /= slope_map.max()

        for step in range(steps):
            perception = self.perceive(state) #32 * 4 = 128
            model_input = torch.cat([perception, static], dim=1) # 128 + 2 = 130
            update = self.w2(F.relu(self.w1(model_input)))

            if use_physics:
                resistance = 1.0 - (slope_map * 3.0)
                update = update * torch.clamp(resistance, 0.0, 1.0)

            if self.training:
                mask = (torch.rand(b, 1, h, w, device=x.device) > 0.5).float()
                update = update * mask

            state = state + update
            forest = torch.min(state[:, 0:1], forest_init).clamp(0, 1)
            state = torch.cat([forest, state[:, 1:]], dim=1)

        return state, state[:, 0:1]

# 2. BASELINES (CNN & RF)
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 1, 3, padding=1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x[:, 0:3])

class RFBaseline:
    def __init__(self): 
        self.model = RandomForestRegressor(n_estimators=50, n_jobs=-1) # Increased estimators for fairness
    def fit(self, X, y): 
        self.model.fit(X, y)
    def predict(self, x_tensor):
        b, c, h, w = x_tensor.shape
        flat = x_tensor[:, 0:3].permute(0, 2, 3, 1).reshape(-1, 3).cpu().numpy()
        return self.model.predict(flat).reshape(b, 1, h, w)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

# --- 1. DEFORESTATION-AWARE LOSS (Fixed Device Bug) ---
class DeforestationLoss(nn.Module):
    """
    Specialized loss for satellite deforestation prediction:
    - Focal loss to handle class imbalance
    - Dice loss for spatial overlap
    - Edge-aware component for boundary detection
    """
    def __init__(self, focal_alpha=0.75, focal_gamma=2.0):
        super().__init__()
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.epsilon = 1e-7
        
        # Sobel for edge detection
        # We register them as buffers so they move with .to(device)
        self.register_buffer('sobel_x', torch.tensor(
            [[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3))
        self.register_buffer('sobel_y', torch.tensor(
            [[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3))

    def focal_loss(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        
        bce = -(target * torch.log(pred + self.epsilon) + 
                (1 - target) * torch.log(1 - pred + self.epsilon))
        
        p_t = torch.where(target == 1, pred, 1 - pred)
        focal_term = (1 - p_t) ** self.focal_gamma
        alpha_t = torch.where(target == 1, self.focal_alpha, 1 - self.focal_alpha)
        
        return (alpha_t * focal_term * bce).mean()

    def dice_loss(self, pred, target):
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        return 1 - (2. * intersection + self.epsilon) / (pred.sum() + target.sum() + self.epsilon)

    def edge_loss(self, pred, target):
        # Compute edges (Now safe because buffers move to GPU)
        pred_gx = F.conv2d(pred, self.sobel_x, padding=1)
        pred_gy = F.conv2d(pred, self.sobel_y, padding=1)
        pred_edge = torch.sqrt(pred_gx**2 + pred_gy**2 + self.epsilon)
        
        target_gx = F.conv2d(target, self.sobel_x, padding=1)
        target_gy = F.conv2d(target, self.sobel_y, padding=1)
        target_edge = torch.sqrt(target_gx**2 + target_gy**2 + self.epsilon)
        
        return F.mse_loss(pred_edge, target_edge)

    def forward(self, pred, target):
        l_focal = self.focal_loss(pred, target)
        l_dice = self.dice_loss(pred, target)
        l_edge = self.edge_loss(pred, target)
        # Weighted combination: 50% Focal, 30% Dice, 20% Edge
        return 0.5 * l_focal + 0.3 * l_dice + 0.2 * l_edge

# --- 2. MULTI-SCALE DEFORESTATION NCA ---
class DeforestationLSD_NCA(nn.Module):
    def __init__(self, in_channels=3, latent_dim=64):
        super().__init__()
        
        # Multi-scale encoder
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64, latent_dim, 3, padding=1), nn.BatchNorm2d(latent_dim), nn.ReLU())
        
        # Multi-scale perception
        self.perception_local = nn.Sequential(nn.Conv2d(latent_dim * 4, 128, 1), nn.BatchNorm2d(128), nn.ReLU())
        self.perception_regional = nn.Sequential(nn.Conv2d(latent_dim * 4, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        
        # Update network
        self.update_net = nn.Sequential(
            nn.Conv2d(256, 256, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 128, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, latent_dim, 1)
        )
        
        # Multi-scale decoder
        self.dec1 = nn.Sequential(nn.Conv2d(latent_dim, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.dec2 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.dec3 = nn.Conv2d(32, 1, 1)
        
        self.register_buffer('sobel_x', torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3))
        self.register_buffer('sobel_y', torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3))

    def perceive(self, z):
        b, c, h, w = z.shape
        gx = F.conv2d(z, self.sobel_x.expand(c, 1, 3, 3), padding=1, groups=c)
        gy = F.conv2d(z, self.sobel_y.expand(c, 1, 3, 3), padding=1, groups=c)
        avg = F.avg_pool2d(z, 3, stride=1, padding=1)
        return torch.cat([z, gx, gy, avg], dim=1)

    def forward_dynamics(self, z, steps):
        states = [z]
        for step in range(steps):
            perception = self.perceive(z)
            local_feat = self.perception_local(perception)
            regional_feat = self.perception_regional(perception)
            combined = torch.cat([local_feat, regional_feat], dim=1)
            
            update = self.update_net(combined)
            if self.training:
                mask = (torch.rand_like(z[:, :1, :, :]) > 0.2).float()
                update = update * mask
            
            z = z + 0.3 * update
            z = torch.clamp(z, -2, 2)
            states.append(z)
        return z, states

    def forward(self, x, steps=32):
        f1 = self.enc1(x)
        f2 = self.enc2(f1)
        z_0 = self.enc3(f2)
        z_final, states = self.forward_dynamics(z_0, steps)
        d1 = self.dec1(z_final)
        d1 = F.interpolate(d1, scale_factor=2, mode='bilinear', align_corners=False)
        d2 = self.dec2(d1)
        pred = self.dec3(d2)
        return torch.sigmoid(pred), z_final, states

# --- 3. TRAINING ENGINE (Fixed) ---
def train_deforestation_lsd_nca(loader, epochs=40, device='cuda'): # Reduced epochs for speed
    first_x, _ = next(iter(loader))
    input_channels = first_x.shape[1]
    print(f"⚡ Detected {input_channels} Input Channels.")
    
    model = DeforestationLSD_NCA(in_channels=input_channels, latent_dim=64).to(device)
    
    # FIX: Ensure criterion is moved to device!
    criterion = DeforestationLoss(focal_alpha=0.75, focal_gamma=2.0).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print(f"🚀 Training Multi-Scale LSD-NCA ({epochs} epochs)...")
    
    for epoch in range(epochs):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            pred, z_final, states = model(x, steps=32)
            
            l_seg = criterion(pred, y)
            l_temporal = F.mse_loss(states[-1], states[-2]) * 0.005
            
            loss = l_seg + l_temporal
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())
            
        scheduler.step()
        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1}: Loss {sum(losses)/len(losses):.4f}")
            
    return model

In [8]:
# --- Cell 6: The "Goldilocks" Training Engine ---
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

def train_nca_model(model, loader, epochs=75):
    # 1. RESTORED WEIGHTS
    # 15.0 was strong enough to learn, but not so strong it broke the model.
    criterion = RobustChangeLoss(change_weight=15.0, false_positive_weight=5.0)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # 2. RESTORED SCHEDULER (Cosine Annealing)
    # This guarantees we land in a minimum by the end.
    # By extending epochs to 75, we keep the LR higher for longer.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"🌲 Training Physics-Informed NCA ({epochs} epochs)...")

    history = []
    best_loss = float('inf')
    model.train()

    for epoch in range(epochs):
        epoch_loss = []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            # Standard Robust Steps
            steps = np.random.randint(32, 48)
            _, pred = model(x, steps=steps)

            loss = criterion(pred, y, x[:, 0:1])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            epoch_loss.append(loss.item())

        avg_loss = np.mean(epoch_loss)
        history.append(avg_loss)

        scheduler.step()

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "nca_best_of_both.pth")

        if epoch % 5 == 0 or epoch == epochs-1:
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"   Epoch {epoch+1}/{epochs}: Loss {avg_loss:.4f} | LR: {curr_lr:.6f}")

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(history, label='Training Loss')
    plt.title("NCA Learning Curve (Restored)")
    plt.xlabel("Epochs"); plt.ylabel("Robust Loss")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

    model.load_state_dict(torch.load("nca_best_of_both.pth"))
    print(f"✅ Best model restored (Loss: {best_loss:.4f}).")
    return model

# Baselines (Standard)
def train_cnn_model(model, loader, epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    print(f"🧠 Training Baseline CNN ({epochs} epochs)...")
    model.train()
    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
    return model

def train_rf_model(loader):
    print("🌳 Training Baseline Random Forest...")
    X_list, y_list = [], []
    for i, (x, y) in enumerate(loader):
        if i > 500: break
        px = x[:, 0:3].permute(0, 2, 3, 1).reshape(-1, 3).numpy()
        py = y.reshape(-1).numpy()
        mask = np.random.rand(len(px)) > 0.8
        X_list.append(px[mask]); y_list.append(py[mask])
    rf = RFBaseline()
    rf.fit(np.concatenate(X_list), np.concatenate(y_list))
    return rf

In [9]:
# --- Cell 7: Final Benchmark ---
def run_expanded_benchmark():
    if 'ds_real' not in globals(): 
        print("⚠️ Data not loaded.")
        return

    print("🧪 Setting up Laboratory...")
    train_len = int(0.8 * len(ds_real))
    test_len = len(ds_real) - train_len
    train_ds, test_ds = random_split(ds_real, [train_len, test_len])
    
    global train_loader, test_loader
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

    # 1. Load Old Model (if exists)
    old_nca = ResearchNCA(channels=32).to(device)
    # (Simplified path logic for brevity)
    if os.path.exists("nca_best.pth"):
        try: old_nca.load_state_dict(torch.load("nca_best_of_both.pth", map_location=device))
        except: pass
    old_nca.eval()

    # 2. Train New Superior Model
    lsd_nca = train_deforestation_lsd_nca(train_loader, epochs=100, device=device)
    lsd_nca.eval()

    # 3. Train Baselines
    print("⚡ Training Baselines...")
    cnn = SimpleCNN().to(device)
    cnn_opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
    for _ in range(5):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.mse_loss(cnn(x), y)
            loss.backward(); cnn_opt.step(); cnn_opt.zero_grad()
            
    rf = RFBaseline()
    X_l, y_l = [], []
    for i, (x, y) in enumerate(train_loader):
        if i > 50: break
        c_dim = min(x.shape[1], 3)
        px = x[:, 0:c_dim].permute(0, 2, 3, 1).reshape(-1, c_dim).numpy()
        py = y.reshape(-1).numpy()
        mask = np.random.rand(len(px)) > 0.95 
        X_l.append(px[mask]); y_l.append(py[mask])
    rf.fit(np.concatenate(X_l), np.concatenate(y_l))

    # 4. Compare
    print("📊 Running Analysis...")
    results = []
    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i >= 50: break
            x_dev, y_cpu = x.to(device), y.numpy()
            
            _, p_old = old_nca(x_dev, steps=32)
            p_lsd, _, _ = lsd_nca(x_dev, steps=32)
            p_cnn = cnn(x_dev)
            
            # SAFE RF PREDICT
            p_rf = rf.predict(x_dev.cpu())
            
            models = [
                (p_old.cpu().numpy(), '1. Standard NCA'), 
                (p_lsd.cpu().numpy(), '2. Multi-Scale NCA'),
                (p_cnn.cpu().numpy(), '3. CNN'), 
                (p_rf, '4. Random Forest')
            ]
            
            for p, name in models:
                m = get_scientific_metrics(y_cpu, p)
                m['Model'] = name
                results.append(m)
                
    df = pd.DataFrame(results)
    print("\n🏆 Final Metrics:")
    print(df.groupby('Model').mean()[['IoU', 'F1', 'Spatial_Structure', 'MSE']])
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df.melt(id_vars="Model"), x="variable", y="value", hue="Model", palette="viridis")
    plt.show()
    return lsd_nca

final_model = run_expanded_benchmark()

🧪 Setting up Laboratory...
⚡ Detected 3 Input Channels.
🚀 Training Multi-Scale LSD-NCA (100 epochs)...
   Epoch 5: Loss 0.2336
   Epoch 10: Loss 0.1508
   Epoch 15: Loss 0.1029
   Epoch 20: Loss 0.0781
   Epoch 25: Loss 0.0639
   Epoch 30: Loss 0.0507
   Epoch 35: Loss 0.0417
   Epoch 40: Loss 0.0372
   Epoch 45: Loss 0.0313
   Epoch 50: Loss 0.0272
   Epoch 55: Loss 0.0231
   Epoch 60: Loss 0.0201
   Epoch 65: Loss 0.0185
   Epoch 70: Loss 0.0161
   Epoch 75: Loss 0.0146
   Epoch 80: Loss 0.0134
   Epoch 85: Loss 0.0134
   Epoch 90: Loss 0.0123
   Epoch 95: Loss 0.0119
   Epoch 100: Loss 0.0119
⚡ Training Baselines...
📊 Running Analysis...

🏆 Final Metrics:
                         IoU        F1  Spatial_Structure       MSE
Model                                                              
1. Standard NCA     0.691794  0.814958           0.875858  0.060111
2. Multi-Scale NCA  0.848334  0.916371           0.922298  0.022998
3. CNN              0.828742  0.904616           0.865633  0.

NameError: name 'sns' is not defined

<Figure size 1000x500 with 0 Axes>

In [10]:
# --- Cell 8: Save & Download Model ---
from IPython.display import FileLink

# 1. Save the 'Final' state (just in case)
final_path = "lsd_nca_final_complete.pth"
torch.save(final_model.state_dict(), final_path)
print(f"✅ Saved final model to: {final_path}")

# 2. Check for the 'Best' model (from training loop)
best_path = "lsd_nca_deforestation_best.pth"
if os.path.exists(best_path):
    print(f"✅ Found 'Best' model (lowest loss) at: {best_path}")
    print("👇 Click below to download the BEST version:")
    display(FileLink(best_path))
else:
    print(f"⚠️ 'Best' model file not found at {best_path}")

print("👇 Click below to download the FINAL version:")
display(FileLink(final_path))


NameError: name 'final_model' is not defined

In [11]:
# --- Cell 8: Experiment - Acquire "Unseen" Data (Pará, Brazil) ---
import os
import ee
import requests

# Ensure EE is initialized
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()

def download_experiment_data():
    filename = "para_experiment_stack.tif"

    # Clean up to ensure fresh download
    if os.path.exists(filename):
        os.remove(filename)

    print("🛰️ Targeting New Region: Pará, Brazil (Trans-Amazonian Highway)...")
    
    # New Coordinates (Approx 50km x 50km)
    # Different geometry than Rondônia
    roi = ee.Geometry.Rectangle([-52.5, -3.5, -52.0, -3.0])

    # --- REUSE EXISTING PIPELINE LOGIC (Must match training data exactly) ---
    # 1. Forest Cover
    gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
    tree_cover = gfc.select('treecover2000').unmask(0)
    loss_year = gfc.select('lossyear').unmask(0)

    # 2. Infrastructure (Roads)
    wc = ee.ImageCollection("ESA/WorldCover/v100").first().select('Map')
    infrastructure = wc.eq(50) 
    infra_100m = infrastructure.reproject(crs='EPSG:4326', scale=100)
    dist_kernel = ee.Kernel.euclidean(radius=20000, units='meters')
    infra_dist = infra_100m.distance(kernel=dist_kernel).unmask(20000)
    road_norm = infra_dist.divide(20000).clamp(0, 1).multiply(-1).add(1)

    # 3. Elevation
    srtm = ee.Image('USGS/SRTMGL1_003').unmask(0)
    elev_norm = srtm.divide(1000).clamp(0, 1)

    # 4. Create Stack
    def get_forest_at_year(y):
        return tree_cover.gt(50).And(loss_year.eq(0).Or(loss_year.gt(y - 2000)))

    forest_2015 = get_forest_at_year(2015).rename('forest_in')
    forest_2020 = get_forest_at_year(2020).rename('forest_out')

    final_stack = ee.Image.cat([forest_2015, road_norm, elev_norm, forest_2020]).float()

    # 5. Download
    print("Requesting download URL for Experiment Data...")
    try:
        url = final_stack.getDownloadURL({
            'scale': 100, 
            'crs': 'EPSG:4326', 
            'region': roi, 
            'format': 'GEO_TIFF'
        })
        print(f"Downloading from: {url}")
        r = requests.get(url, stream=True)
        with open(filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("✅ Experiment Data Acquired: para_experiment_stack.tif")
    except Exception as e:
        print(f"Export Failed: {e}")

# Run the download
download_experiment_data()

🛰️ Targeting New Region: Pará, Brazil (Trans-Amazonian Highway)...
Requesting download URL for Experiment Data...
Export Failed: Project 'projects/32555940559.apps.googleusercontent.com' not found or deleted.


In [12]:
# --- Cell 9: Load Experiment Data ---
# Reuse the class from earlier in the notebook
ds_experiment = RealDeforestationDataset(
    "para_experiment_stack.tif", 
    patch_size=64, 
    stride=32, 
    augment=False # Crucial: No augmentation for testing
)

# Create a loader
exp_loader = DataLoader(ds_experiment, batch_size=1, shuffle=True)
print(f"🧪 Experiment Set: {len(ds_experiment)} patches ready for testing.")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [13]:
# --- Cell 10: Generalization Test (Final Fix) ---
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

def ensure_baselines_exist():
    global cnn, rf
    if 'cnn' not in globals() or 'rf' not in globals():
        print("⚠️ Baselines not found. Retraining on Rondônia data...")
        
        # 1. Retrain CNN
        cnn = SimpleCNN().to(device)
        cnn_opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
        print("   - Training CNN (5 epochs)...")
        for _ in range(5):
            for x, y in train_loader: 
                x, y = x.to(device), y.to(device)
                loss = F.mse_loss(cnn(x), y)
                loss.backward(); cnn_opt.step(); cnn_opt.zero_grad()
        
        # 2. Retrain RF
        rf = RFBaseline()
        print("   - Training Random Forest...")
        X_l, y_l = [], []
        for i, (x, y) in enumerate(train_loader):
            if i > 50: break
            c_dim = min(x.shape[1], 3)
            # Flatten for fitting
            px = x[:, 0:c_dim].permute(0, 2, 3, 1).reshape(-1, c_dim).numpy()
            py = y.reshape(-1).numpy()
            mask = np.random.rand(len(px)) > 0.95 
            X_l.append(px[mask]); y_l.append(py[mask])
        rf.fit(np.concatenate(X_l), np.concatenate(y_l))
        print("✅ Baselines Restored.")
    else:
        print("✅ Baselines found in memory.")

def run_generalization_experiment(nca_model, loader):
    print("\n⚡ STARTING GENERALIZATION TEST: PARÁ REGION")
    ensure_baselines_exist()
    
    iterator = iter(loader)
    
    # Show 3 Examples
    for i in range(3):
        try:
            x, y_true = next(iterator)
        except StopIteration:
            break
            
        x_dev = x.to(device)
        y_true_np = y_true.numpy()
        
        # --- 1. PREDICT ---
        # A. NCA (FIXED UNPACKING HERE)
        with torch.no_grad():
            # The model returns: (prediction, latent_state, history)
            # We want the FIRST element
            pred_nca, _, _ = nca_model(x_dev, steps=32)
            pred_nca = pred_nca.cpu().numpy()
            
        # B. CNN
        with torch.no_grad():
            pred_cnn = cnn(x_dev).cpu().numpy()
            
        # C. RF
        # Pass 4D Tensor on CPU (Batch, Channel, Height, Width)
        pred_rf = rf.predict(x.cpu()) 

        # --- 2. VISUALIZE ---
        input_rgb = np.zeros((64, 64, 3))
        # Visualizing Input: Red=Roads, Green=Forest (approx)
        if x.shape[1] >= 2:
            input_rgb[:, :, 1] = x[0, 0].numpy() * 0.8 # Forest
            input_rgb[:, :, 0] = x[0, 1].numpy()       # Roads
        
        fig, ax = plt.subplots(1, 5, figsize=(20, 5))
        cmap = 'Greens'
        
        # Helper for IoU title
        def get_iou(y, p):
            return get_scientific_metrics(y, p)['IoU']
        
        ax[0].imshow(input_rgb)
        ax[0].set_title(f"Pará Input {i+1}\n(Red=Roads)")
        
        ax[1].imshow(y_true_np[0, 0], cmap=cmap, vmin=0, vmax=1)
        ax[1].set_title("Ground Truth")
        
        ax[2].imshow(pred_nca[0, 0], cmap=cmap, vmin=0, vmax=1)
        ax[2].set_title(f"NCA Prediction\nIoU: {get_iou(y_true_np, pred_nca):.3f}")
        
        ax[3].imshow(pred_cnn[0, 0], cmap=cmap, vmin=0, vmax=1)
        ax[3].set_title(f"CNN Prediction\nIoU: {get_iou(y_true_np, pred_cnn):.3f}")
        
        ax[4].imshow(pred_rf[0, 0, 0], cmap=cmap, vmin=0, vmax=1)
        ax[4].set_title(f"RF Prediction\nIoU: {get_iou(y_true_np, pred_rf):.3f}")
        
        for a in ax: a.axis('off')
        plt.tight_layout()
        plt.show()

# Run the test
if 'final_model' in globals():
    run_generalization_experiment(final_model, exp_loader)
else:
    print("⚠️ 'final_model' not found. Run Cell 7 first.")

⚠️ 'final_model' not found. Run Cell 7 first.


In [15]:
MODEL_PATH = "/kaggle/working/lsd_nca_deforestation.pth"
torch.save(final_model.state_dict(), MODEL_PATH)
print("Model saved to", MODEL_PATH)


NameError: name 'final_model' is not defined